In [ ]:
# @title Установка и настройка Ollama (Kaggle, 2x T4 GPU)
#!/usr/bin/env python3
"""
Скрипт 1/3: Установка и запуск Ollama на Kaggle (2x NVIDIA T4)
Отличия от Colab-версии:
  - Нет Google Drive — модели хранятся в /kaggle/working/ (сессионное хранилище)
  - CUDA_VISIBLE_DEVICES=0,1 — Ollama использует обе T4 (32 GB VRAM суммарно)
Размер модели задаётся переменной OLLAMA_MODEL ниже.
"""

import os
import subprocess
import time
import sys

# ─────────────────────────────────────────────
# Конфигурация
# ─────────────────────────────────────────────
OLLAMA_MODELS_DIR = '/kaggle/working/mafia/ollama_models'
OLLAMA_PORT       = 11434

# ── Выбор модели Ollama ──────────────────────────────────────
# 🧪 Тест (минимальная):  "gemma3:1b"   ~0.8 GB VRAM
# 🚀 Рабочая:             "gemma3:4b"   ~3   GB VRAM
# 💪 Продвинутая:         "qwen2.5:7b"  ~5   GB VRAM
# 🔥 Максимальная:        "gemma3:12b"  ~8   GB VRAM
OLLAMA_MODEL      = "gemma3:1b"
REQUIRED_MODELS   = [OLLAMA_MODEL]


def print_step(step_num, message):
    print(f"\n{'='*60}")
    print(f"ШАГ {step_num}: {message}")
    print(f"{'='*60}\n")


def run_command(cmd, check=True, shell=False):
    try:
        if isinstance(cmd, str) and not shell:
            cmd = cmd.split()
        result = subprocess.run(cmd, shell=shell, check=check, capture_output=True, text=True)
        return result
    except subprocess.CalledProcessError as e:
        print(f"❌ Ошибка выполнения команды: {e}")
        if e.stdout: print(f"STDOUT: {e.stdout}")
        if e.stderr: print(f"STDERR: {e.stderr}")
        raise


def check_gpus():
    """Проверяет наличие GPU и выводит информацию"""
    print_step(1, "Проверка GPU")
    result = run_command(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], check=False)
    if result.returncode == 0:
        gpus = result.stdout.strip().splitlines()
        print(f"✅ Найдено GPU: {len(gpus)}")
        for i, gpu in enumerate(gpus):
            print(f"   GPU {i}: {gpu.strip()}")
        if len(gpus) < 2:
            print("⚠️  Ожидалось 2 GPU — убедитесь что выбран Accelerator: GPU T4 x2")
        return True
    else:
        print("❌ nvidia-smi недоступен — GPU не найден")
        return False


def setup_ollama_environment():
    print_step(2, "Настройка окружения Ollama (2x T4)")
    # Используем обе T4 — Ollama автоматически распределит модель
    os.environ['CUDA_VISIBLE_DEVICES']     = '0,1'
    os.environ['OLLAMA_MODELS']            = OLLAMA_MODELS_DIR
    os.environ['OLLAMA_HOST']              = '0.0.0.0'
    # Позволяем держать несколько моделей загруженными (по одной на каждую GPU)
    os.environ['OLLAMA_MAX_LOADED_MODELS'] = '2'
    os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
    print(f"✅ CUDA_VISIBLE_DEVICES=0,1  (обе T4 активны)")
    print(f"✅ OLLAMA_HOST=0.0.0.0       (разрешены внешние подключения)")
    print(f"✅ OLLAMA_MAX_LOADED_MODELS=2")
    print(f"✅ Директория моделей: {OLLAMA_MODELS_DIR}")
    return True


def install_ollama():
    print_step(3, "Установка Ollama")
    if os.path.exists('/usr/local/bin/ollama'):
        print("✅ Ollama уже установлена")
        return True
    print("📦 Установка зависимостей...")
    run_command("apt-get update -qq", shell=True)
    run_command("apt-get install -y -qq zstd curl", shell=True)
    print("📦 Установка Ollama...")
    run_command("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    result = run_command(["ollama", "--version"], check=False)
    if result.returncode == 0:
        print(f"✅ Ollama установлена: {result.stdout.strip()}")
        return True
    print("❌ Ollama установлена, но не работает")
    return False


def start_ollama_service():
    print_step(4, "Запуск Ollama сервиса")
    result = run_command(["pgrep", "-f", "ollama serve"], check=False)
    if result.returncode == 0:
        print("🔄 Перезапуск Ollama с актуальными настройками...")
        run_command(["pkill", "-f", "ollama serve"], check=False)
        time.sleep(2)
    print(f"🚀 Запуск Ollama (CUDA_VISIBLE_DEVICES=0,1)...")
    env = os.environ.copy()
    env['OLLAMA_HOST']              = '0.0.0.0'
    env['OLLAMA_MODELS']            = OLLAMA_MODELS_DIR
    env['CUDA_VISIBLE_DEVICES']     = '0,1'
    env['OLLAMA_MAX_LOADED_MODELS'] = '2'
    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    for i in range(20):
        time.sleep(1)
        result = run_command(
            ["curl", "-s", f"http://localhost:{OLLAMA_PORT}/api/tags"],
            check=False
        )
        if result.returncode == 0:
            print(f"✅ Ollama запущена на порту {OLLAMA_PORT}")
            return True
        print(f"   Ожидание... ({i+1}/20)")
    print("⚠️  Ollama запущена, но HTTP-проверка не прошла")
    return True


def check_model_installed(model_name):
    """Проверяет что модель установлена и занимает нормальный объём"""
    try:
        result = run_command(["ollama", "list"], check=False)
        if result.returncode != 0 or model_name not in result.stdout:
            return False
        blobs_dir = os.path.join(OLLAMA_MODELS_DIR, "blobs")
        if not os.path.exists(blobs_dir):
            return False
        total_size = sum(
            os.path.getsize(os.path.join(blobs_dir, f))
            for f in os.listdir(blobs_dir)
            if os.path.isfile(os.path.join(blobs_dir, f))
        )
        size_gb = total_size / (1024 ** 3)
        # Минимальный порог зависит от модели: 1b ~0.5 GB, 4b ~2 GB, 12b ~7 GB
        min_gb = 0.3
        if size_gb < min_gb:
            print(f"   ⚠️  Модель битая: {size_gb:.2f} GB (ожидается >{min_gb} GB) — удаляю")
            run_command(["ollama", "rm", model_name], check=False)
            return False
        print(f"   Размер моделей: {size_gb:.1f} GB — OK")
        return True
    except Exception as e:
        print(f"   Ошибка проверки модели: {e}")
        return False


def download_models():
    print_step(5, "Загрузка моделей Ollama")
    for model in REQUIRED_MODELS:
        print(f"\n📥 Проверка модели: {model}")
        if check_model_installed(model):
            print(f"✅ Модель {model} уже установлена")
        else:
            print(f"📥 Загрузка {model}... (может занять несколько минут)")
            run_command(["ollama", "pull", model], check=True)
            print(f"✅ Модель {model} загружена")
    print("\n✅ Все модели готовы")
    return True


def warmup_model(model_name):
    """Прогревает модель — загружает веса в VRAM обоих GPU"""
    print(f"🔥 Прогрев модели {model_name} (загрузка весов в VRAM 2x T4)...")
    result = run_command(
        ["curl", "-s", "-X", "POST", f"http://localhost:{OLLAMA_PORT}/api/chat",
         "-H", "Content-Type: application/json",
         "-d", f'{{"model":"{model_name}","messages":[{{"role":"user","content":"hi"}}],"stream":false}}'],
        check=False
    )
    if result.returncode == 0:
        print(f"✅ Модель прогрета — следующие запросы будут быстрыми")
    else:
        print(f"⚠️  Прогрев не удался: {result.stderr[:200]}")


def verify_ollama():
    print_step(6, "Проверка работы Ollama")
    result = run_command(["ollama", "list"], check=False)
    if result.returncode == 0:
        print("✅ Ollama работает")
        print("\n📋 Установленные модели:")
        print(result.stdout)
        # Показываем распределение по GPU
        gpu_result = run_command(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total",
             "--format=csv,noheader"], check=False
        )
        if gpu_result.returncode == 0:
            print("📊 Использование VRAM:")
            for i, line in enumerate(gpu_result.stdout.strip().splitlines()):
                print(f"   GPU {i}: {line.strip()}")
        for model in REQUIRED_MODELS:
            warmup_model(model)
        return True
    print("❌ Ollama не отвечает")
    return False


def main():
    print("\n" + "="*60)
    print("🚀 ШАГ 1/3: УСТАНОВКА OLLAMA (Kaggle, 2x T4)")
    print("="*60 + "\n")

    steps = [
        ("Проверка GPU",       check_gpus),
        ("Настройка окружения", setup_ollama_environment),
        ("Установка Ollama",   install_ollama),
        ("Запуск сервиса",     start_ollama_service),
        ("Загрузка моделей",   download_models),
        ("Проверка",           verify_ollama),
    ]

    for name, func in steps:
        try:
            if not func():
                print(f"⚠️  Шаг '{name}' завершился с предупреждением")
        except Exception as e:
            print(f"❌ Шаг '{name}' завершился с ошибкой: {e}")
            print("   Продолжаю выполнение...")

    print("\n" + "="*60)
    print("✅ OLLAMA ГОТОВА")
    print("="*60)
    print(f"\n   Ollama слушает на: http://localhost:{OLLAMA_PORT}")
    print(f"   Теперь запустите ячейку 2 (Whisper)")
    print(f"   Затем запустите   ячейку 3 (Server)\n")


if __name__ == "__main__":
    main()

In [ ]:
# @title Установка WhisperX (Kaggle)
#!/usr/bin/env python3

import os
import shutil
import subprocess
import sys

KAGGLE_CACHE_ROOT = "/kaggle/working/.cache"
TORCH_HOME = f"{KAGGLE_CACHE_ROOT}/torch"
HF_HOME = f"{KAGGLE_CACHE_ROOT}/huggingface"
HF_HUB_CACHE = f"{HF_HOME}/hub"

# ── Выбор модели WhisperX ────────────────────────────────────
# 🧪 Тест (минимальная):  "base"    ~140 MB,  HF: guillaumekln/faster-whisper-base
# 🚀 Рабочая:             "small"   ~460 MB,  HF: guillaumekln/faster-whisper-small
# 💪 Продвинутая:         "medium"  ~1.5 GB,  HF: guillaumekln/faster-whisper-medium
WHISPER_SIZE     = "base"
WHISPER_HF_REPO  = f"guillaumekln/faster-whisper-{WHISPER_SIZE}"
WHISPER_DIR      = f"/kaggle/working/models/{WHISPER_SIZE}"


def print_step(step_num, message):
    print(f"\n{'='*60}")
    print(f"ШАГ {step_num}: {message}")
    print(f"{'='*60}\n")


def run(cmd, check=True):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-500:])
    if check and result.returncode != 0:
        print(f"❌ Ошибка:\n{result.stderr[-500:]}")
        raise SystemExit(1)
    return result


def check_gpu():
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        for line in result.stdout.splitlines():
            if 'Tesla' in line or 'T4' in line or 'A100' in line or 'L4' in line:
                print(f"✅ GPU: {line.strip()}")
                break
        return True
    print("⚠️  GPU не найден — whisperX будет работать на CPU (медленно)")
    return False


def _clean_ld_paths(env):
    """Убираем из LD_LIBRARY_PATH / LD_PRELOAD пути к TensorFlow/XLA .so,
    чтобы динамический линковщик не загружал их CUDA-фабрики,
    которые конфликтуют с PyTorch (std::bad_alloc)."""
    _TOXIC = ("tensorflow", "/xla/", "tf_framework", "libtf_")
    for var in ("LD_LIBRARY_PATH", "LD_PRELOAD"):
        val = env.get(var, "")
        if not val:
            continue
        clean = [
            p for p in val.split(":")
            if not any(tok in p.lower() for tok in _TOXIC)
        ]
        env[var] = ":".join(clean)
    return env


def build_subprocess_env():
    os.makedirs(TORCH_HOME, exist_ok=True)
    os.makedirs(HF_HOME, exist_ok=True)
    os.makedirs(HF_HUB_CACHE, exist_ok=True)

    env = os.environ.copy()
    env["TORCH_HOME"] = TORCH_HOME
    env["HF_HOME"] = HF_HOME
    env["HUGGINGFACE_HUB_CACHE"] = HF_HUB_CACHE
    env.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
    # Подавляем конфликт TensorFlow ↔ PyTorch CUDA-библиотек на Kaggle
    env["TF_CPP_MIN_LOG_LEVEL"] = "3"
    env["CUDA_MODULE_LOADING"] = "LAZY"
    env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
    env["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
    _clean_ld_paths(env)
    return env


def build_cpu_test_env():
    env = build_subprocess_env()
    # Для sanity-check полностью скрываем GPU от подпроцесса,
    # чтобы сторонние зависимости не пытались инициализировать CUDA.
    env["CUDA_VISIBLE_DEVICES"] = ""
    env["OMP_NUM_THREADS"] = "1"
    env["MKL_NUM_THREADS"] = "1"
    env["NUMEXPR_NUM_THREADS"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    return env


def python_network_bootstrap():
    return f"""
import os
import shutil
import urllib.request
from http.cookiejar import CookieJar

os.environ["TORCH_HOME"] = "{TORCH_HOME}"
os.environ["HF_HOME"] = "{HF_HOME}"
os.environ["HUGGINGFACE_HUB_CACHE"] = "{HF_HUB_CACHE}"
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

opener = urllib.request.build_opener(
    urllib.request.HTTPRedirectHandler(),
    urllib.request.HTTPCookieProcessor(CookieJar()),
)
opener.addheaders = [("User-Agent", "Mozilla/5.0")]
urllib.request.install_opener(opener)
"""


def clear_vad_cache():
    """Удаляет только кэш silero VAD, если он скачался битым."""
    targets = [
        f"{TORCH_HOME}/hub/checkpoints",
        f"{TORCH_HOME}/hub/checkpoints/silero_vad.jit",
        f"{TORCH_HOME}/hub/checkpoints/silero_vad.onnx",
        f"{TORCH_HOME}/hub/snakers4_silero-vad_master",
        f"{TORCH_HOME}/hub/snakers4_silero-vad_main",
        f"{TORCH_HOME}/hub/trusted_list",
        f"{TORCH_HOME}/hub/trustlist.json",
        os.path.expanduser("~/.cache/torch/hub/checkpoints"),
        os.path.expanduser("~/.cache/torch/hub/snakers4_silero-vad_master"),
        os.path.expanduser("~/.cache/torch/hub/snakers4_silero-vad_main"),
        os.path.expanduser("~/.cache/torch/hub/trusted_list"),
        os.path.expanduser("~/.cache/torch/hub/trustlist.json"),
    ]

    removed = False
    for path in targets:
        if os.path.isdir(path):
            shutil.rmtree(path, ignore_errors=True)
            removed = True
        elif os.path.exists(path):
            os.remove(path)
            removed = True

    if removed:
        print("🧹 Очищен локальный кэш silero VAD")


def unload_ollama_models():
    """Выгружает все модели Ollama из VRAM/RAM, чтобы освободить память
    для установки и теста WhisperX. Модели загрузятся обратно автоматически
    при первом запросе через ячейку 3."""
    import json
    import urllib.request
    try:
        r = urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5)
        data = json.loads(r.read())
        models = data.get("models", [])
        if not models:
            return
        for m in models:
            name = m.get("name", "")
            if not name:
                continue
            req = urllib.request.Request(
                "http://localhost:11434/api/generate",
                data=json.dumps({"model": name, "keep_alive": 0}).encode(),
                headers={"Content-Type": "application/json"},
            )
            try:
                urllib.request.urlopen(req, timeout=30)
            except Exception:
                pass
            print(f"🔄 Выгружена модель Ollama «{name}» из VRAM")
        import time
        time.sleep(2)  # даём Ollama освободить память
    except Exception:
        pass  # Ollama может быть не запущена — OK


def install_whisperx():
    print_step(1, "Установка WhisperX")

    result = subprocess.run(
        ['python', '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅ PyTorch уже есть: {result.stdout.strip()}")
    else:
        print("📦 Устанавливаю PyTorch...")
        run("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q")

    print("📦 Обновляю WhisperX до свежей версии...")
    run("pip uninstall -y whisperx >/dev/null 2>&1 || true", check=False)
    run("pip install --no-cache-dir --upgrade git+https://github.com/m-bain/whisperX.git -q")

    print("📦 Устанавливаю pyannote.audio (диаризация)...")
    run("pip install --upgrade pyannote.audio -q")

    print("📦 Устанавливаю совместимые зависимости (без отката свежего WhisperX)...")
    run("pip install --upgrade numpy==1.26.4 -q")

    # TensorFlow предустановлен на Kaggle и его CUDA .so конфликтуют с PyTorch
    # (std::bad_alloc при совместной инициализации cuDNN/cuBLAS/cuFFT).
    # WhisperX не использует TF — безопасно удаляем.
    print("🧹 Удаляю TensorFlow (конфликт CUDA-библиотек с PyTorch)...")
    run(
        "pip uninstall -y tensorflow tensorflow-gpu tensorflow-estimator "
        "tensorflow-io-gcs-filesystem keras tf-keras tensorboard "
        "tensorboard-data-server 2>/dev/null || true",
        check=False,
    )

    result = subprocess.run(
        ['python', '-c', 'import importlib.metadata as m; print("whisperx", m.version("whisperx")); print("faster-whisper", m.version("faster-whisper")); print("ctranslate2", m.version("ctranslate2"))'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅ WhisperX установлен и готов:\n{result.stdout.strip()}")
        return True
    else:
        print(f"❌ WhisperX не установился:\n{result.stderr[:300]}")
        return False


def check_hf_token():
    print_step(2, "Проверка HuggingFace токена")

    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    if token:
        print("✅ HF_TOKEN найден в переменных окружения")
        return token

    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        token = secrets.get_secret("HF_TOKEN")
        if token:
            os.environ['HF_TOKEN'] = token
            print("✅ HF_TOKEN получен из Kaggle Secrets")
            return token
    except Exception:
        pass

    print("⚠️  HF_TOKEN не найден — диаризация будет недоступна")
    print("   1. Получите токен: https://huggingface.co/settings/tokens")
    print("   2. Примите условия: https://huggingface.co/pyannote/speaker-diarization-3.1")
    print("   3. Добавьте токен в Kaggle Secrets с именем HF_TOKEN")
    return None


def download_model():
    print_step(3, f"Загрузка модели Whisper {WHISPER_SIZE}")

    model_path = WHISPER_DIR

    if os.path.exists(model_path) and os.listdir(model_path):
        print(f"✅ Модель уже скачана: {model_path}")
        return model_path

    print(f"📥 Скачиваю модель {WHISPER_HF_REPO}, подождите...")
    download_code = f"""
from huggingface_hub import snapshot_download
path = snapshot_download(
    repo_id="{WHISPER_HF_REPO}",
    local_dir="{model_path}"
)
print(f"✅ Модель сохранена: {{path}}")
"""
    result = subprocess.run(
        ['python', '-c', download_code],
        capture_output=True, text=True, timeout=600
    )
    if result.returncode == 0:
        print(result.stdout)
        return model_path
    else:
        print(f"❌ Ошибка скачивания:\n{result.stderr[-500:]}")
        return None


def preload_vad():
    """✅ Предзагрузка silero VAD через torch.hub — обходит проблему 301 редиректа"""
    print_step(4, "Предзагрузка silero VAD")

    clear_vad_cache()

    vad_code = f"""
{python_network_bootstrap()}
import torch

torch.hub.set_dir("{TORCH_HOME}")
print("📥 Загружаю silero VAD...")
model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=True,
    trust_repo=True
)
print("✅ silero VAD загружен")
"""
    result = subprocess.run(
        ['python', '-c', vad_code],
        capture_output=True, text=True, timeout=120,
        env=build_subprocess_env()
    )
    if result.returncode == 0:
        print(result.stdout)
        return True
    else:
        print(f"❌ VAD не загрузился:\n{result.stderr[-500:]}")
        return False


def _write_test_script(model_path, force_cpu=False):
    """Записывает тест-скрипт во временный .py файл.
    Файл вместо python -c — меньше потребление RAM при fork().
    """
    device_line = (
        'test_device = "cpu"'
        if force_cpu
        else 'test_device = "cuda" if torch.cuda.is_available() else "cpu"'
    )
    code = f"""\
import os, sys, shutil, traceback, urllib.request
from http.cookiejar import CookieJar

os.environ["TORCH_HOME"] = "{TORCH_HOME}"
os.environ["HF_HOME"] = "{HF_HOME}"
os.environ["HUGGINGFACE_HUB_CACHE"] = "{HF_HUB_CACHE}"
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

opener = urllib.request.build_opener(
    urllib.request.HTTPRedirectHandler(),
    urllib.request.HTTPCookieProcessor(CookieJar()),
)
opener.addheaders = [("User-Agent", "Mozilla/5.0")]
urllib.request.install_opener(opener)

# Блокируем TF-импорт (подстраховка)
class _TFBlocker:
    def find_module(self, name, path=None):
        if name == "tensorflow" or name.startswith("tensorflow."):
            return self
        return None
    def load_module(self, name):
        raise ImportError(f"blocked: {{name}}")
sys.meta_path.insert(0, _TFBlocker())

print("Старт WhisperX ASR-подпроцесса...", flush=True)
import torch
import whisperx.asr as whisperx_asr
import whisperx.vad as whisperx_vad
from whisperx.audio import load_audio

{device_line}
compute_types = ["int8", "int8_float16", "float16"] if test_device == "cuda" else ["int8"]
print(f"Устройство={{test_device}}, compute_type={{compute_types}}", flush=True)

torch.hub.set_dir("{TORCH_HOME}")

_orig_load_vad = whisperx_vad.load_vad_model
def _patched_load_vad(*a, **kw):
    try:
        return _orig_load_vad(*a, **kw)
    except RuntimeError as e:
        if "SHA256 checksum" not in str(e):
            raise
        for p in [
            "{TORCH_HOME}/hub/checkpoints",
            "{TORCH_HOME}/hub/snakers4_silero-vad_master",
            "{TORCH_HOME}/hub/snakers4_silero-vad_main",
        ]:
            shutil.rmtree(p, ignore_errors=True)
        return _orig_load_vad(*a, **kw)

whisperx_vad.load_vad_model = _patched_load_vad
whisperx_asr.load_vad_model = _patched_load_vad

model = None
for ct in compute_types:
    print(f"Пробую compute_type={{ct}}...", flush=True)
    try:
        try:
            model = whisperx_asr.load_model(
                "{model_path}", test_device,
                compute_type=ct, language="ru", vad_method="silero",
            )
        except TypeError:
            model = whisperx_asr.load_model(
                "{model_path}", test_device,
                compute_type=ct, language="ru",
            )
        print(f"✅ Модель загружена (compute_type={{ct}})", flush=True)
        break
    except Exception as exc:
        print(f"⚠️  {{ct}}: {{type(exc).__name__}}: {{exc}}", flush=True)

if model is None:
    sys.exit(1)

audio = load_audio("/tmp/test_whisperx.wav")
result = model.transcribe(audio, language="ru", batch_size=1)
print(f"Транскрипция: {{result}}")
print("✅ WhisperX работает!")
"""
    script_path = "/tmp/_whisperx_test.py"
    with open(script_path, "w") as f:
        f.write(code)
    return script_path


def _write_import_test_script():
    """Лёгкий тест: только проверка импортов (без загрузки модели).
    Потребляет минимум RAM."""
    code = f"""\
import os, sys
os.environ["TORCH_HOME"] = "{TORCH_HOME}"
os.environ["HF_HOME"] = "{HF_HOME}"
os.environ["HUGGINGFACE_HUB_CACHE"] = "{HF_HUB_CACHE}"

class _TFBlocker:
    def find_module(self, name, path=None):
        if name == "tensorflow" or name.startswith("tensorflow."):
            return self
        return None
    def load_module(self, name):
        raise ImportError(f"blocked: {{name}}")
sys.meta_path.insert(0, _TFBlocker())

print("Проверяю импорты...", flush=True)
import torch
print(f"  torch {{torch.__version__}}, CUDA={{torch.cuda.is_available()}}", flush=True)
import whisperx
import importlib.metadata as meta
print(f"  whisperx {{meta.version('whisperx')}}", flush=True)
print(f"  faster-whisper {{meta.version('faster-whisper')}}", flush=True)
print(f"  ctranslate2 {{meta.version('ctranslate2')}}", flush=True)

model_path = "{WHISPER_DIR}"
if os.path.isdir(model_path) and os.listdir(model_path):
    print(f"  модель: {{model_path}} ({{len(os.listdir(model_path))}} файлов)", flush=True)
else:
    print("❌ Модель не найдена!", flush=True)
    sys.exit(1)

print("✅ Импорты и модель OK", flush=True)
"""
    script_path = "/tmp/_whisperx_import_test.py"
    with open(script_path, "w") as f:
        f.write(code)
    return script_path


def test_whisperx(model_path):
    import gc
    print_step(5, "Тест WhisperX")

    subprocess.run(
        'ffmpeg -f lavfi -i anullsrc=r=16000:cl=mono -t 3 /tmp/test_whisperx.wav -y -loglevel quiet',
        shell=True
    )

    # Освобождаем RAM родительского процесса перед fork()
    gc.collect()

    # ── Попытка 1: полный GPU-тест (скрипт из .py файла) ──────────────────
    script = _write_test_script(model_path, force_cpu=False)
    result = subprocess.run(
        ['python', script],
        capture_output=True, text=True, timeout=180,
        env=build_subprocess_env()
    )
    if result.returncode == 0:
        print(result.stdout)
        print("✅ Тест пройден (GPU)")
        return True

    gpu_output = (result.stdout + "\n" + result.stderr).strip()
    has_oom = any(k in gpu_output for k in (
        "std::bad_alloc", "OutOfMemoryError", "CUDA out of memory",
        "Cannot allocate memory", "MemoryError", "Killed",
    ))

    if not has_oom:
        print(f"❌ Тест не прошёл:\n{gpu_output[-1500:]}")
        return False

    # ── Попытка 2: лёгкий import-only тест (без загрузки модели) ──────────
    print("⚠️  Полный тест упал с нехваткой памяти — проверяю импорты...")
    gc.collect()
    import_script = _write_import_test_script()
    result = subprocess.run(
        ['python', import_script],
        capture_output=True, text=True, timeout=60,
        env=build_cpu_test_env()
    )
    if result.returncode == 0:
        print(result.stdout)
        print("✅ Импорты и модель на месте (полный тест пропущен — нехватка RAM)")
        print("   Транскрипция будет работать при запуске сервера "
              "(модель загрузится в отдельном процессе)")
        return True

    output = (result.stdout + "\n" + result.stderr).strip()
    print(f"❌ Даже импорты не прошли:\n{output[-1500:]}")
    return False


def save_config(hf_token, model_path):
    config_path = "/kaggle/working/whisper_config.txt"
    import json
    config = {
        "engine": "whisperx",
        "model": model_path,
        "language": "ru",
        "hf_token": hf_token or "",
        "diarization": bool(hf_token),
    }
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    print(f"✅ Конфиг сохранён: {config_path}")


def main():
    print("\n" + "="*60)
    print("🚀 ШАГ 2/3: УСТАНОВКА WHISPERX (Kaggle)")
    print("="*60 + "\n")

    # Если ячейка 1 загрузила Ollama модели — выгружаем, чтобы не было OOM
    unload_ollama_models()

    has_gpu = check_gpu()

    if not install_whisperx():
        sys.exit(1)

    hf_token = check_hf_token()

    model_path = download_model()
    if not model_path:
        print("❌ Не удалось скачать модель")
        sys.exit(1)

    if not preload_vad():
        print("⚠️  VAD не загрузился — тест может не пройти")

    if not test_whisperx(model_path):
        print("⚠️  Тест не прошёл — проверьте ошибки выше")
        sys.exit(1)

    save_config(hf_token, model_path)

    print("\n" + "="*60)
    print("✅ WHISPERX ГОТОВ")
    print("="*60)
    print(f"\n   Движок:      WhisperX ({WHISPER_SIZE})")
    print(f"   Модель:      {model_path}")
    print(f"   HF repo:     {WHISPER_HF_REPO}")
    print(f"   Диаризация:  {'✅ включена' if hf_token else '❌ нет HF_TOKEN'}")
    print(f"   GPU:         {'✅' if has_gpu else '❌ CPU'}")
    print(f"\n   Теперь запустите ячейку 3 (Server)\n")


if __name__ == "__main__":
    main()

In [ ]:
# @title Единый Flask-прокси + ngrok туннель (Kaggle)
#!/usr/bin/env python3
"""
Скрипт 3/3: Единый Flask-прокси + ngrok туннель для Ollama и WhisperX (Kaggle)
Маршруты:
  /api/*          → Ollama (localhost:11434)
  /whisper/file   → WhisperX (GPU транскрипция)
  /health         → общий healthcheck + статус GPU
"""

import os
import subprocess
import time
import sys
import json
import tempfile
import urllib.request
from flask import Flask, request, Response, jsonify, stream_with_context

# ─────────────────────────────────────────────
# Конфигурация
# ─────────────────────────────────────────────
OLLAMA_PORT         = 11434
FLASK_PORT          = 5000
LANGUAGE            = "ru"
WHISPER_CONFIG_FILE = "/kaggle/working/whisper_config.txt"
NGROK_AUTH_TOKEN    = '3AoDR90lG7GchjAwnPxAIdAOisx_5HFZ54HKyxfYTHcuNqMYa'

KAGGLE_CACHE_ROOT   = "/kaggle/working/.cache"
TORCH_HOME          = f"{KAGGLE_CACHE_ROOT}/torch"
HF_HOME             = f"{KAGGLE_CACHE_ROOT}/huggingface"
HF_HUB_CACHE        = f"{HF_HOME}/hub"

app = Flask(__name__)

# Глобальные переменные для WhisperX модели (загружаются один раз при старте)
_whisperx_model = None
_whisperx_config = None


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def run_command(cmd, check=True, shell=False):
    if isinstance(cmd, str) and not shell:
        cmd = cmd.split()
    return subprocess.run(cmd, shell=shell, check=check, capture_output=True, text=True)


def load_whisper_config():
    """Загружает JSON-конфиг WhisperX, созданный ячейкой 2."""
    global _whisperx_config
    if not os.path.exists(WHISPER_CONFIG_FILE):
        return None
    with open(WHISPER_CONFIG_FILE) as f:
        try:
            _whisperx_config = json.load(f)
            return _whisperx_config
        except (json.JSONDecodeError, ValueError):
            return None


def init_whisperx():
    """Загружает WhisperX модель один раз при старте сервера."""
    global _whisperx_model

    config = load_whisper_config()
    if not config:
        print("⚠️  WhisperX конфиг не найден — запустите ячейку 2")
        return False

    model_path = config.get("model", "")
    if not os.path.isdir(model_path):
        print(f"❌ Модель WhisperX не найдена: {model_path}")
        return False

    os.environ["TORCH_HOME"] = TORCH_HOME
    os.environ["HF_HOME"] = HF_HOME
    os.environ["HUGGINGFACE_HUB_CACHE"] = HF_HUB_CACHE
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

    print(f"📥 Загрузка WhisperX модели: {model_path}...")
    import torch
    import whisperx

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "int8"
    torch.hub.set_dir(TORCH_HOME)

    # GPU 1 — для WhisperX, GPU 0 остаётся для Ollama
    device_index = 0
    if device == "cuda" and torch.cuda.device_count() > 1:
        device_index = 1
        print(f"   Используем GPU {device_index} для WhisperX (GPU 0 → Ollama)")

    try:
        _whisperx_model = whisperx.load_model(
            model_path,
            device,
            device_index=device_index,
            compute_type=compute_type,
            language=config.get("language", "ru"),
        )
    except TypeError:
        _whisperx_model = whisperx.load_model(
            model_path,
            device,
            compute_type=compute_type,
        )

    gpu_label = f"{device}:{device_index}" if device == "cuda" else device
    print(f"✅ WhisperX загружен (device={gpu_label}, compute_type={compute_type})")
    return True


def ollama_is_running():
    try:
        result = run_command(
            ["curl", "-s", "--max-time", "2", f"http://localhost:{OLLAMA_PORT}/api/tags"],
            check=False
        )
        return result.returncode == 0
    except:
        return False


# ─────────────────────────────────────────────
# Маршруты: Ollama-прокси
# ─────────────────────────────────────────────

@app.route('/api/<path:path>', methods=['GET', 'POST', 'PUT', 'DELETE'])
def ollama_proxy(path):
    """Проксирует все /api/* запросы к Ollama (2x T4)"""
    import requests as req_lib

    target_url = f"http://localhost:{OLLAMA_PORT}/api/{path}"
    if request.query_string:
        target_url += '?' + request.query_string.decode()

    headers = {
        k: v for k, v in request.headers
        if k.lower() not in ('host', 'content-length', 'transfer-encoding', 'connection')
    }

    try:
        resp = req_lib.request(
            method=request.method,
            url=target_url,
            headers=headers,
            data=request.get_data(),
            stream=True,
            timeout=300,
        )

        def generate():
            for chunk in resp.iter_content(chunk_size=1):
                if chunk:
                    yield chunk

        return Response(
            stream_with_context(generate()),
            status=resp.status_code,
            content_type=resp.headers.get('Content-Type', 'application/json'),
            headers={
                'X-Accel-Buffering': 'no',
                'Cache-Control': 'no-cache',
            }
        )
    except Exception as e:
        return jsonify({"error": f"Ollama proxy error: {str(e)}"}), 502


# ─────────────────────────────────────────────
# Маршруты: WhisperX
# ─────────────────────────────────────────────

@app.route('/whisper/file', methods=['POST'])
def whisper_file():
    """Принимает аудио файл, возвращает транскрипцию в NDJSON через WhisperX"""
    global _whisperx_model

    if _whisperx_model is None:
        return jsonify({"error": "WhisperX model not loaded — run cell 2 and restart cell 3"}), 500

    if 'audio' not in request.files:
        return jsonify({"error": "no audio file provided"}), 400

    audio_file = request.files['audio']
    language   = request.form.get('language', LANGUAGE)

    with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as tmp:
        audio_file.save(tmp.name)
        tmp_path = tmp.name

    try:
        from whisperx.audio import load_audio

        audio = load_audio(tmp_path)
        result = _whisperx_model.transcribe(audio, language=language, batch_size=8)

        segments = result.get("segments", [])

        def generate():
            for seg in segments:
                text = seg.get("text", "").strip()
                if text:
                    yield json.dumps({"text": text}, ensure_ascii=False) + "\n"

        return Response(generate(), mimetype='application/x-ndjson')

    except Exception as e:
        return jsonify({"error": f"WhisperX error: {str(e)}"}), 500
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)


# ─────────────────────────────────────────────
# Маршруты: Health
# ─────────────────────────────────────────────

@app.route('/health', methods=['GET'])
def health():
    gpu_info = []
    try:
        r = run_command(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
            check=False
        )
        if r.returncode == 0:
            gpu_info = [line.strip() for line in r.stdout.strip().splitlines()]
    except:
        pass

    config = _whisperx_config or {}
    return jsonify({
        "status": "ok",
        "gpus": gpu_info,
        "ollama": {
            "running": ollama_is_running(),
            "url": f"http://localhost:{OLLAMA_PORT}",
        },
        "whisper": {
            "engine": "whisperx",
            "model": config.get("model", "not configured"),
            "model_loaded": _whisperx_model is not None,
            "language": config.get("language", LANGUAGE),
            "diarization": config.get("diarization", False),
        },
    })


# ─────────────────────────────────────────────
# Ngrok
# ─────────────────────────────────────────────

def install_ngrok():
    if os.path.exists('/usr/local/bin/ngrok'):
        print("✅ ngrok уже установлен")
        return True
    print("📦 Установка ngrok...")
    run_command(
        "curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null "
        "&& echo 'deb https://ngrok-agent.s3.amazonaws.com buster main' | tee /etc/apt/sources.list.d/ngrok.list "
        "&& apt update -qq && apt install ngrok -y -qq",
        shell=True, check=False
    )
    if os.path.exists('/usr/local/bin/ngrok'):
        print("✅ ngrok установлен")
        return True
    print("❌ Не удалось установить ngrok")
    return False


def setup_ngrok_tunnel():
    print("\n" + "="*60)
    print("Настройка ngrok туннеля")
    print("="*60)

    if not install_ngrok():
        return None

    run_command(["pkill", "-f", "ngrok"], check=False)
    time.sleep(1)

    if NGROK_AUTH_TOKEN:
        print("🔐 Авторизация ngrok...")
        run_command(["ngrok", "config", "add-authtoken", NGROK_AUTH_TOKEN], check=False)
    else:
        print("⚠️  NGROK_AUTH_TOKEN не задан")

    print(f"🚇 Запуск туннеля на порт {FLASK_PORT}...")
    subprocess.Popen(
        ["ngrok", "http", str(FLASK_PORT), "--log=stdout"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(4)

    try:
        req = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=5)
        data = json.loads(req.read().decode())
        tunnels = data.get("tunnels", [])
        if tunnels:
            public_url = tunnels[0].get("public_url", "")
            if public_url:
                return public_url
    except Exception as e:
        print(f"⚠️  Не удалось получить URL туннеля: {e}")
        print("   Проверьте вручную: http://localhost:4040")

    return None


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    print("\n" + "="*60)
    print("🚀 ШАГ 3/3: ЕДИНЫЙ СЕРВЕР (OLLAMA + WHISPERX) — Kaggle 2x T4")
    print("="*60 + "\n")

    # Установка flask/requests если нужно
    try:
        import flask, requests
    except ImportError:
        print("📦 Установка flask и requests...")
        run_command("pip install -q flask requests", shell=True, check=False)

    # Проверка зависимостей
    errors = []

    if not ollama_is_running():
        errors.append("❌ Ollama не запущена — выполните ячейку 1")
    else:
        print(f"✅ Ollama работает на порту {OLLAMA_PORT}")

    if not init_whisperx():
        errors.append("❌ WhisperX не загружен — выполните ячейку 2")
    else:
        config = _whisperx_config or {}
        print(f"✅ WhisperX модель: {config.get('model', '?')}")

    # Статус GPU
    try:
        r = run_command(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
            check=False
        )
        if r.returncode == 0:
            print("\n📊 Использование VRAM (2x T4):")
            for i, line in enumerate(r.stdout.strip().splitlines()):
                print(f"   GPU {i}: {line.strip()}")
    except:
        pass

    if errors:
        print("\n" + "="*60)
        print("🛑 ЗАПУСК ПРЕРВАН — не все компоненты готовы:")
        print("="*60)
        for e in errors:
            print(f"   {e}")
        print("\n💡 Порядок запуска: ячейка 1 → ячейка 2 → ячейка 3")
        return

    # Запуск ngrok
    public_url = setup_ngrok_tunnel()
    base_url = public_url or f"http://localhost:{FLASK_PORT}"

    print("\n" + "="*60)
    print("✅ СЕРВЕР ГОТОВ")
    print("="*60)
    print(f"\n🌐 Локальный URL:  http://localhost:{FLASK_PORT}")
    if public_url:
        print(f"🌐 Публичный URL:  {public_url}")
    else:
        print(f"⚠️  ngrok туннель не создан")

    print(f"\n📋 Обновите config.json:")
    print(f'   "ollama_base_url": "{base_url}"')
    print(f'   "whisper_url":     "{base_url}/whisper/file"')

    print(f"\n📋 Endpoints:")
    print(f"   GET  {base_url}/health")
    print(f"   ANY  {base_url}/api/*          → Ollama (2x T4)")
    print(f"   POST {base_url}/whisper/file   → WhisperX (GPU)")

    print(f"\n⚠️  Для API-запросов через ngrok добавляйте заголовок:")
    print(f'   ngrok-skip-browser-warning: true')
    print(f"\n⚠️  Kaggle-сессия должна оставаться активной!\n")

    print("🚀 Запуск Flask сервера...")
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, threaded=True,
            use_reloader=False)


if __name__ == "__main__":
    main()